In [5]:
import os
import glob
import numpy as np
import scipy.ndimage as ndimage
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. ATTENTION U-NET MODEL ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.size()[2:] != x1.size()[2:]:
            g1 = F.interpolate(g1, size=x1.size()[2:], mode='bilinear', align_corners=True)
        relu = self.relu(g1 + x1)
        alpha = self.psi(relu)
        return x * alpha

class UpAttention(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
        self.ag = AttentionBlock(F_g=in_channels // 2, F_l=in_channels // 2, F_int=in_channels // 4)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x2_attended = self.ag(g=x1, x=x2)
        x = torch.cat([x2_attended, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
        return self.conv(x)

class AttentionUNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=4, bilinear=False):
        super(AttentionUNet, self).__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)

        self.up1 = UpAttention(1024, 512 // factor, bilinear)
        self.up2 = UpAttention(512, 256 // factor, bilinear)
        self.up3 = UpAttention(256, 128 // factor, bilinear)
        self.up4 = UpAttention(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# --- 2. HELPER FUNCTIONS ---
def keep_largest_connected_component_3d(pred_volume_3d, classes=[1, 2, 3]):
    cleaned_volume = np.copy(pred_volume_3d)
    for cls in classes:
        binary_mask = (pred_volume_3d == cls)
        if not np.any(binary_mask):
            continue
        labeled_array, num_features = ndimage.label(binary_mask)
        if num_features <= 1:
            continue
        component_sizes = ndimage.sum(binary_mask, labeled_array, range(1, num_features + 1))
        largest_component_label = np.argmax(component_sizes) + 1
        smaller_components_mask = (labeled_array > 0) & (labeled_array != largest_component_label)
        cleaned_volume[smaller_components_mask & (cleaned_volume == cls)] = 0
    return cleaned_volume

@torch.no_grad()
def predict_patient_volume(model, volume_2d_stack, device):
    model.eval()
    num_slices = volume_2d_stack.shape[2]
    pred_slices = []
    for slice_idx in range(num_slices):
        slice_2d = volume_2d_stack[:, :, slice_idx]
        tensor_in = torch.tensor(slice_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        logits = model(tensor_in)
        pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        pred_slices.append(pred_mask)
    return np.stack(pred_slices, axis=2).astype(np.uint8)

def export_split_predictions(model, split_name, img_dir, mask_dir, target_out_dir, device):
    os.makedirs(target_out_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(img_dir, "*.npy")))
    print(f"\nProcessing {len(files)} volumes for split '{split_name}' -> '{target_out_dir}'...")
    
    for filepath in tqdm(files, desc=f"Exporting {split_name} predictions"):
        filename = os.path.basename(filepath)
        volume = np.load(filepath)
        
        # 1. Inference & 3D Post-Processing
        raw_pred = predict_patient_volume(model, volume, device)
        clean_pred = keep_largest_connected_component_3d(raw_pred, classes=[1, 2, 3])
        
        # 2. Save 3D Mask Array (.npy)
        mask_save_name = filename.replace(".npy", "_att_pred.npy")
        mask_save_path = os.path.join(target_out_dir, mask_save_name)
        np.save(mask_save_path, clean_pred)
        
        # 3. Save Multi-Panel Visualization Overlay (.png)
        gt_filename = filename.replace(".npy", "_gt.npy")
        gt_path = os.path.join(mask_dir, gt_filename)
        has_gt = os.path.exists(gt_path)
        gt_vol = np.load(gt_path) if has_gt else None
        
        slice_idx = volume.shape[2] // 2
        img_slice = volume[:, :, slice_idx]
        pred_slice = clean_pred[:, :, slice_idx]
        
        if has_gt and gt_vol is not None:
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            axes[0].imshow(img_slice, cmap='gray')
            axes[0].set_title(f"Raw MRI ({filename[:-4]}, Slice {slice_idx})", fontsize=11, fontweight='bold')
            axes[0].axis('off')
            
            gt_slice = gt_vol[:, :, slice_idx]
            axes[1].imshow(img_slice, cmap='gray')
            masked_gt = np.ma.masked_where(gt_slice == 0, gt_slice)
            axes[1].imshow(masked_gt, cmap='rainbow', alpha=0.5, vmin=1, vmax=3)
            axes[1].set_title("Ground Truth Mask Overlay", fontsize=11, fontweight='bold')
            axes[1].axis('off')
            
            axes[2].imshow(img_slice, cmap='gray')
            masked_pred = np.ma.masked_where(pred_slice == 0, pred_slice)
            axes[2].imshow(masked_pred, cmap='rainbow', alpha=0.5, vmin=1, vmax=3)
            axes[2].set_title("Attention U-Net Prediction Overlay", fontsize=11, fontweight='bold')
            axes[2].axis('off')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
            axes[0].imshow(img_slice, cmap='gray')
            axes[0].set_title(f"Raw MRI ({filename[:-4]}, Slice {slice_idx})", fontsize=11, fontweight='bold')
            axes[0].axis('off')
            
            axes[1].imshow(img_slice, cmap='gray')
            masked_pred = np.ma.masked_where(pred_slice == 0, pred_slice)
            axes[1].imshow(masked_pred, cmap='rainbow', alpha=0.5, vmin=1, vmax=3)
            axes[1].set_title("Attention U-Net Prediction Overlay", fontsize=11, fontweight='bold')
            axes[1].axis('off')
            
        plt.tight_layout()
        img_save_name = filename.replace(".npy", "_overlay.png")
        img_save_path = os.path.join(target_out_dir, img_save_name)
        plt.savefig(img_save_path, dpi=150, bbox_inches='tight')
        plt.close()

# --- 3. EXPORT PIPELINE ---
def export_stage_1_predictions_subfolders():
    base_dir = r"C:\D\ACDC"
    weights_path = os.path.join(base_dir, "training", "best_attention_unet_model.pth")
    
    stage1_pred_dir = os.path.join(base_dir, "stage_1_predictions")
    train_out_dir = os.path.join(stage1_pred_dir, "train_data")
    test_out_dir = os.path.join(stage1_pred_dir, "test_data")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("=========================================================")
    print(" EXPORTING STAGE 1 PREDICTIONS & OVERLAYS TO SUBFOLDERS ")
    print("=========================================================")
    print(f" Train Output Dir:  {train_out_dir}")
    print(f" Test Output Dir:   {test_out_dir}")
    print(f" Device:            {device}")
    print("---------------------------------------------------------")
    
    model = AttentionUNet(n_channels=1, n_classes=4, bilinear=False).to(device)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f"[Loaded]: Attention U-Net weights from '{weights_path}'!\n")
    
    # 1. Export Training Predictions (100 patients / 200 volumes)
    train_img_dir = os.path.join(base_dir, "preprocessed data", "images", "train")
    train_mask_dir = os.path.join(base_dir, "preprocessed data", "masks", "train")
    export_split_predictions(model, "train", train_img_dir, train_mask_dir, train_out_dir, device)
    
    # 2. Export Testing Predictions (50 patients / 100 volumes)
    test_img_dir = os.path.join(base_dir, "preprocessed data", "images", "test")
    test_mask_dir = os.path.join(base_dir, "preprocessed data", "masks", "test")
    export_split_predictions(model, "test", test_img_dir, test_mask_dir, test_out_dir, device)
    
    print("\n=========================================================")
    print(" SUCCESS: Exported all training & testing predictions!")
    print("=========================================================")

export_stage_1_predictions_subfolders()

 EXPORTING STAGE 1 PREDICTIONS & OVERLAYS TO SUBFOLDERS 
 Train Output Dir:  C:\D\ACDC\stage_1_predictions\train_data
 Test Output Dir:   C:\D\ACDC\stage_1_predictions\test_data
 Device:            cuda
---------------------------------------------------------


C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_10748\3531343829.py:232: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_lo

[Loaded]: Attention U-Net weights from 'C:\D\ACDC\training\best_attention_unet_model.pth'!


Processing 200 volumes for split 'train' -> 'C:\D\ACDC\stage_1_predictions\train_data'...


Exporting train predictions: 100%|██████████| 200/200 [03:42<00:00,  1.11s/it]



Processing 100 volumes for split 'test' -> 'C:\D\ACDC\stage_1_predictions\test_data'...


Exporting test predictions: 100%|██████████| 100/100 [01:29<00:00,  1.12it/s]


 SUCCESS: Exported all training & testing predictions!
